# LaDe joint route + ETA model on Colab GPU
1. **Runtime > Change runtime type > T4 GPU**, then run the cells top to bottom.
2. Upload `colab/lade_colab.zip` to Google Drive folder **`MyDrive/lade/`** first (one time).
3. Results (`joint_*.json`, model `.pt`) are written straight to `MyDrive/lade/results/`, so they survive disconnects.
   Download that folder into the project's `results/` afterwards.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/lade'

In [ ]:
# copy to local disk (much faster than reading from Drive), unzip, keep results on Drive
!mkdir -p $DRIVE/results/models
!cp $DRIVE/lade_colab.zip /content/ && cd /content && unzip -qo lade_colab.zip && rm lade_colab.zip
!rm -rf /content/lade-route-eta/results && ln -s $DRIVE/results /content/lade-route-eta/results
%cd /content/lade-route-eta
!pip -q install -r colab/requirements_colab.txt
import torch, pandas; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| pandas', pandas.__version__)

## Check: smoke test (tiny data, 1 epoch, CPU) — should finish in a few minutes and print metrics

In [ ]:
!python scripts/train_joint.py --dataset pickup --smoke

## Real run (GPU)
Start with **one seed** and check time per epoch and RAM (the RAM bar at top right).
Free Colab has ~12 GB RAM; if it crashes while building data, lower `--frac` (e.g. 0.1).

In [ ]:
!python scripts/train_joint.py --dataset pickup --frac 0.25 --seed 0

## All 5 seeds, both datasets (CLAUDE.md: report mean ± std over 5 seeds)
Already-finished runs are skipped, so you can re-run this cell after a disconnect.

In [ ]:
import os
for ds in ['pickup', 'delivery']:
    for s in range(5):
        if os.path.exists(f'results/joint_{ds}_h64_l3_lam1.0_s{s}.json'):
            print('done, skipping', ds, s); continue
        !python scripts/train_joint.py --dataset {ds} --frac 0.25 --seed {s}

In [ ]:
!ls -la results/ results/models/